|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 7:</h2>|<h1>Modern vLLM<h1>|
|<h2>Section:</h2>|<h1>Speculative decoding<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: verify, reject, and prove nothing changed<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

Implement the verification step, and prove it does not change the model.

The speedup you already measured. This is about the other half: a speculative
decoder that is 3x faster and samples from a slightly different distribution
is not a faster model, it is a different one.

In [ ]:
### run this cell

V = 8                                   # a tiny vocabulary
target = np.array([0.40, 0.25, 0.15, 0.10, 0.05, 0.03, 0.01, 0.01])
draft  = np.array([0.30, 0.30, 0.20, 0.10, 0.05, 0.03, 0.01, 0.01])
print('target and draft disagree, on purpose')

# Exercise 1: accept or resample

In [ ]:
def verify(p_target, p_draft, drafted, u):
  """Accept the drafted token, or resample. Returns (token, accepted)."""
  if u < min(1.0, p_target[drafted] / p_draft[drafted]):
    return drafted, True
  residual = np.maximum(0.0, p_target - p_draft)
  residual = residual / residual.sum()
  return int(rng.choice(len(residual), p=residual)), False

acc = 0
for _ in range(10):
  d = int(rng.choice(V, p=draft))
  tokv, ok = verify(target, draft, d, rng.random())
  acc += ok
print(f'{acc}/10 drafts accepted')

# Exercise 2: is the output distribution unchanged?

Two hundred thousand draws against a distribution you know exactly. This is
the only test that can catch a subtly wrong verifier.

In [ ]:
N = 200_000
counts = np.zeros(V)
accepted = 0
for _ in range(N):
  d = int(rng.choice(V, p=draft))
  t, ok = verify(target, draft, d, rng.random())
  counts[t] += 1; accepted += ok
emp = counts/N

print(f"{'token':>6} {'target':>8} {'sampled':>9} {'error':>8}")
for i,(a,b) in enumerate(zip(target, emp)):
  print(f'{i:>6} {a:>8.3f} {b:>9.3f} {b-a:>+8.4f}')
print(f'\nmax error {np.abs(emp-target).max():.4f}, acceptance {accepted/N:.1%}')

# Exercise 3: now use a draft model that knows nothing

Uniform over the vocabulary. Predict both numbers before you run it.

In [ ]:
terrible = np.full(V, 1.0/V)        # a draft model that knows nothing

counts = np.zeros(V); accepted = 0
for _ in range(N):
  d = int(rng.choice(V, p=terrible))
  t, ok = verify(target, terrible, d, rng.random())
  counts[t] += 1; accepted += ok
emp = counts/N

print(f'uniform draft: acceptance {accepted/N:.1%}, '
      f'max distribution error {np.abs(emp-target).max():.4f}')
print('\nslower, and still exactly the right distribution.')

### The property that makes this shippable

Exercise 3 is the point. A draft model that is pure noise gives you a
low acceptance rate and therefore no speedup, and the output
distribution is still exactly the target's, to sampling error.

That is unusual. Most approximations trade accuracy for speed and you
have to decide how much you will give up. This one trades **speed for
speed**: a better draft is faster, a worse draft is slower, and neither
changes what the model says.

So you can ship a draft model you are not sure about. The worst case is
that you wasted the compute, not that you silently changed every answer
your users get.

Two details that are easy to get wrong:

- **The residual must be clamped at zero before normalising.** Tokens
  the draft over-weights contribute nothing on rejection; they were
  already over-represented among the acceptances.
- **`min(1, ...)`**, not the raw ratio. Where the target likes a token
  more than the draft does, you accept it every time, and the extra
  mass arrives through the residual path.

    ./vc guide 17